# 🗺️ Rumsey Map OCR — Detection Model Fine-tuning
Run each cell **top to bottom**. Training takes **4-6 hours** on a T4 GPU.

> **Before running:** `Runtime → Change runtime type → T4 GPU`

### What you need on Google Drive first:
```
My Drive/
└── Rumsey_OCR/
    ├── train_data/
    │   └── det/            ← upload this from your laptop
    │       ├── train_list.txt
    │       ├── val_list.txt
    │       └── train_images/
    └── det_icdar_finetune.yml   ← upload this config file
```


## Step 1 — Verify GPU


In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout or '❌ No GPU — go to Runtime → Change runtime type → T4 GPU')


## Step 2 — Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/Rumsey_OCR'

# Verify required files are on Drive
checks = {
    'Training data':  os.path.join(DRIVE_ROOT, 'train_data/det/train_list.txt'),
    'Val data':       os.path.join(DRIVE_ROOT, 'train_data/det/val_list.txt'),
    'Config file':    os.path.join(DRIVE_ROOT, 'det_icdar_finetune.yml'),
}
all_ok = True
for name, path in checks.items():
    exists = os.path.exists(path)
    status = '✅' if exists else '❌  MISSING'
    print(f'  {status}  {name}: {path}')
    if not exists: all_ok = False
if not all_ok:
    print('\n⚠️  Upload missing files to Drive before continuing!')
else:
    print('\n✅ All required files found!')


## Step 3 — Install PaddlePaddle GPU
*(Takes ~2 min — run once per session)*


In [ ]:
!pip install paddlepaddle-gpu==2.6.1.post120 \
    -f https://www.paddlepaddle.org.cn/whl/linux/mkl/avx/stable.html -q

import paddle
print(f'PaddlePaddle: {paddle.__version__}')
print(f'GPU available: {paddle.is_compiled_with_cuda()}')


## Step 4 — Get PaddleOCR Toolkit
Clones the official PaddleOCR training tools. *(No need to upload — pulled from PaddlePaddle directly)*


In [ ]:
import os

PADDLEOCR_DIR = '/content/PaddleOCR'

if not os.path.exists(PADDLEOCR_DIR):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git {PADDLEOCR_DIR} -q
    print('Cloned PaddleOCR')
else:
    print('PaddleOCR already present')

# Install requirements
!pip install -r {PADDLEOCR_DIR}/requirements.txt -q
print('Requirements installed!')


## Step 5 — Setup Workspace


In [ ]:
import os, shutil

DRIVE_ROOT   = '/content/drive/MyDrive/Rumsey_OCR'
PADDLEOCR_DIR = '/content/PaddleOCR'
WORK_DIR      = '/content/workspace'
os.makedirs(WORK_DIR, exist_ok=True)

# Link training data from Drive into workspace
DET_DATA_SRC = os.path.join(DRIVE_ROOT, 'train_data/det')
DET_DATA_DST = os.path.join(WORK_DIR, 'train_data/det')
os.makedirs(os.path.dirname(DET_DATA_DST), exist_ok=True)
if not os.path.exists(DET_DATA_DST):
    os.symlink(DET_DATA_SRC, DET_DATA_DST)
    print(f'Linked train_data/det from Drive')

# Copy config into workspace (and fix paths to be absolute)
CONFIG_SRC = os.path.join(DRIVE_ROOT, 'det_icdar_finetune.yml')
CONFIG_DST = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')
shutil.copy2(CONFIG_SRC, CONFIG_DST)

# Link output directory to Drive for REAL-TIME checkpoint saving
# This ensures that if Colab disconnects, your progress is saved on Drive!
DRIVE_OUT_DIR = os.path.join(DRIVE_ROOT, 'output/det_finetune')
LOCAL_OUT_DIR = os.path.join(WORK_DIR, 'output/det_finetune')
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(LOCAL_OUT_DIR), exist_ok=True)

if not os.path.exists(LOCAL_OUT_DIR):
    os.symlink(DRIVE_OUT_DIR, LOCAL_OUT_DIR)
    print(f'✅ Output directory symlinked to Drive: {DRIVE_OUT_DIR}')

# Patch paths in config to be absolute
with open(CONFIG_DST) as f:
    cfg = f.read()
cfg = cfg.replace('../train_data/det', DET_DATA_DST)
cfg = cfg.replace('../output/det_finetune', LOCAL_OUT_DIR)
with open(CONFIG_DST, 'w') as f:
    f.write(cfg)
print('Config ready at:', CONFIG_DST)
print('Workspace ready!')


## Step 6 — Download Pretrained Detection Model


In [ ]:
import os
WORK_DIR    = '/content/workspace'
PRETRAINED  = os.path.join(WORK_DIR, 'pretrained/en_PP-OCRv4_det_train')
TAR_FILE    = os.path.join(WORK_DIR, 'pretrained/en_PP-OCRv4_det_train.tar')

if not os.path.exists(PRETRAINED):
    os.makedirs(os.path.join(WORK_DIR, 'pretrained'), exist_ok=True)
    print('Downloading pretrained model...')
    URL = 'https://paddleocr.bj.bcebos.com/PP-OCRv4/english/en_PP-OCRv4_det_train.tar'
    !wget -q --show-progress -O {TAR_FILE} {URL}
    if os.path.exists(TAR_FILE):
        print('Extracting...')
        !tar -xf {TAR_FILE} -C {os.path.join(WORK_DIR, 'pretrained')}
        print('Extracted successfully!')
    else:
        print('❌ Download failed!')
else:
    print('✅ Pretrained model already present')

# Patch config with pretrained model path
CONFIG_DST = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')
with open(CONFIG_DST) as f:
    cfg = f.read()
cfg = cfg.replace('../pretrained/en_PP-OCRv4_det_train/best_accuracy',
                  os.path.join(PRETRAINED, 'best_accuracy'))
with open(CONFIG_DST, 'w') as f:
    f.write(cfg)
print('Config updated with pretrained model path')


## Step 7 — Verify Training Data


In [ ]:
WORK_DIR = '/content/workspace'
import os

for fname in ['train_list.txt', 'val_list.txt']:
    path = os.path.join(WORK_DIR, 'train_data/det', fname)
    if os.path.exists(path):
        with open(path) as f: n = sum(1 for _ in f)
        print(f'✅ {fname}: {n:,} images')
    else:
        print(f'❌ MISSING: {path}')


## Step 8 — Train Detection Model 🔥
This is the main cell. Takes **4-6 hours**.
> The session stays alive as long as this cell is running.

> **Tip:** Keep your browser tab open (or use Colab Pro for longer sessions).


In [ ]:
WORK_DIR      = '/content/workspace'
PADDLEOCR_DIR = '/content/PaddleOCR'
CONFIG        = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')

import os
os.chdir(PADDLEOCR_DIR)

!python tools/train.py -c {CONFIG}


## Step 9 — Export Trained Model


In [ ]:
WORK_DIR      = '/content/workspace'
PADDLEOCR_DIR = '/content/PaddleOCR'
CONFIG        = os.path.join(WORK_DIR, 'det_icdar_finetune.yml')
BEST_MODEL    = os.path.join(WORK_DIR, 'output/det_finetune/best_accuracy')
INFER_OUT     = os.path.join(WORK_DIR, 'output/det_inference_finetuned')

import os
os.chdir(PADDLEOCR_DIR)

!python tools/export_model.py \
    -c {CONFIG} \
    -o Global.pretrained_model={BEST_MODEL} \
       Global.save_inference_dir={INFER_OUT}

print('Export complete!')


## Step 10 — Save Everything to Drive ✅
Saves checkpoints and the final inference model back to your Google Drive.


In [ ]:
import shutil, os
DRIVE_ROOT = '/content/drive/MyDrive/Rumsey_OCR'
WORK_DIR   = '/content/workspace'

DRIVE_OUT = os.path.join(DRIVE_ROOT, 'output')
os.makedirs(DRIVE_OUT, exist_ok=True)

# Save training checkpoints
src_ckpt = os.path.join(WORK_DIR, 'output/det_finetune')
dst_ckpt = os.path.join(DRIVE_OUT, 'det_finetune')
if os.path.exists(src_ckpt):
    shutil.copytree(src_ckpt, dst_ckpt, dirs_exist_ok=True)
    print(f'✅ Checkpoints saved to Drive')

# Save inference model
src_inf = os.path.join(WORK_DIR, 'output/det_inference_finetuned')
dst_inf = os.path.join(DRIVE_OUT, 'det_inference_finetuned')
if os.path.exists(src_inf):
    shutil.copytree(src_inf, dst_inf, dirs_exist_ok=True)
    print(f'✅ Inference model saved to Drive')

print(f'\nAll outputs at: {DRIVE_OUT}')
print('Download det_inference_finetuned/ to your laptop when ready.')


## ✅ All Done!

After the notebook completes:
1. Download `output/det_inference_finetuned/` from Drive to your laptop
2. Place it in `Rumsey_Map_OCR/output/det_inference_finetuned/`
3. Run `step4_evaluate_and_infer.py` on your laptop to see the improved results

**Expected improvement: 27% → 50-60%+ recall** 🎯
